# Estructuras base

Compara hash con búsqueda lineal, y KD-Tree con KNN por fuerza bruta. Ejecuta todas las celdas. Este notebook contiene su código y no usa los archivos de `test/`.


In [ ]:
RUN_MODE = "quick"  # "quick" mide 1K, 10K y 100K. "full" añade 1M y 10M.
SEED = 42
DATASET_FILE = ""  # Ruta a 2019-Oct.csv. Vacío: muestra sintética solo para validar.

print("Modo:", RUN_MODE)
print("Semilla:", SEED)
print("CSV:", DATASET_FILE or "sintético, solo validación")


In [ ]:
import platform
import time
import tracemalloc
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "01_filtrado_probabilistico.ipynb").exists():
    ROOT = ROOT / "pruebas_algoritmos"

def sizes(mode):
    return [1_000, 10_000, 100_000] if mode == "quick" else [1_000, 10_000, 100_000, 1_000_000, 10_000_000]

def repetitions(n, mode):
    if mode == "quick":
        return 7
    return 3 if n >= 1_000_000 else 100

def prepare_output(folder):
    results, figures = folder / "resultados", folder / "figuras"
    results.mkdir(parents=True, exist_ok=True)
    figures.mkdir(parents=True, exist_ok=True)
    return results, figures

def resolve_data(data):
    if not data:
        return None
    candidate = Path(data)
    options = [candidate, Path.cwd() / candidate, ROOT / candidate, ROOT.parent / candidate]
    found = next((option for option in options if option.is_file()), None)
    if found is None:
        raise FileNotFoundError(f"No existe el CSV indicado: {data}")
    return found

def load_events(data, n, seed):
    found = resolve_data(data)
    if found:
        cols = ["event_time", "event_type", "product_id", "price", "user_id", "user_session", "brand", "category_code"]
        frame = pd.read_csv(found, nrows=n, usecols=lambda column: column in cols)
        return frame, "REES46 CSV"
    rng = np.random.default_rng(seed)
    products = max(300, min(n // 8, 20_000))
    users = max(200, min(n // 5, 50_000))
    frame = pd.DataFrame({
        "event_time": pd.date_range("2019-10-01", periods=n, freq="s"),
        "event_type": rng.choice(["view", "cart", "purchase"], size=n, p=[0.83, 0.12, 0.05]),
        "product_id": rng.integers(1_000_000, 1_000_000 + products, size=n),
        "price": np.round(rng.lognormal(4.2, 0.75, n), 2),
        "user_id": rng.integers(1, users, size=n),
        "user_session": rng.integers(1, max(20, n // 4), size=n).astype(str),
        "brand": rng.choice(["samsung", "apple", "sony", "unknown"], size=n),
        "category_code": rng.choice(["electronics.phone", "appliances.kitchen", "computers"], size=n),
    })
    return frame, "SINTÉTICO (solo validación; no usar en el informe)"

def measure(action, reps):
    timings = []
    for _ in range(reps):
        start = time.perf_counter()
        action()
        timings.append((time.perf_counter() - start) * 1_000)
    tracemalloc.start()
    action()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return float(np.mean(timings)), float(np.std(timings)), peak / 1024 ** 2

def record(rows, **fields):
    rows.append({
        "experimento": fields["experiment"], "algoritmo": fields["algorithm"], "n": fields["n"],
        "metrica": fields["metric"], "valor": fields["value"], "tiempo_promedio_ms": fields["time_ms"],
        "desviacion_ms": fields["std_ms"], "memoria_pico_mb": fields["memory_mb"],
        "repeticiones": fields["reps"], "semilla": fields["seed"],
        "plataforma": platform.platform(), "fuente": fields["source"],
    })

def save(rows, name, output):
    results, _ = prepare_output(output)
    path = results / f"{name}.csv"
    pd.DataFrame(rows).to_csv(path, index=False)
    print(f"Tabla: {path}")
    return path

def line_chart(frame, title, filename, output, value="tiempo_promedio_ms", log=True):
    _, figures = prepare_output(output)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for algorithm, group in frame.groupby("algoritmo"):
        group = group.sort_values("n")
        if log:
            group = group[group[value] > 0]
        if group.empty:
            continue
        ax.plot(group["n"], group[value], marker="o", label=algorithm)
    ax.set(title=title, xlabel="Elementos", ylabel=value.replace("_", " "))
    if log:
        ax.set_xscale("log")
        ax.set_yscale("log")
    ax.grid(alpha=0.3, which="both")
    ax.legend()
    fig.tight_layout()
    path = figures / filename
    fig.savefig(path, dpi=160)
    plt.show()
    plt.close(fig)
    print(f"Figura: {path}")
    return path

import heapq
def exact_knn(vectors: np.ndarray, ids: np.ndarray, query: np.ndarray, k: int) -> list[int]:
    distance = ((vectors - query) ** 2).sum(axis=1)
    return ids[np.argsort(distance)[:k]].tolist()


class KDTree:
    def __init__(self, vectors: np.ndarray, ids: np.ndarray):
        self.vectors, self.ids = vectors, ids
        self.root = self._build(np.arange(len(ids)))

    def _build(self, positions: np.ndarray):
        if not len(positions):
            return None
        axis = int(np.var(self.vectors[positions], axis=0).argmax())
        ordered = positions[np.argsort(self.vectors[positions, axis])]
        middle = len(ordered) // 2
        return axis, ordered[middle], self._build(ordered[:middle]), self._build(ordered[middle + 1:])

    def query(self, vector: np.ndarray, k: int) -> list[int]:
        heap: list[tuple[float, int]] = []
        def visit(node):
            if node is None:
                return
            axis, pos, left, right = node
            squared = float(((self.vectors[pos] - vector) ** 2).sum())
            heapq.heappush(heap, (-squared, pos))
            if len(heap) > k:
                heapq.heappop(heap)
            near, far = (left, right) if vector[axis] < self.vectors[pos, axis] else (right, left)
            visit(near)
            if len(heap) < k or (vector[axis] - self.vectors[pos, axis]) ** 2 < -heap[0][0]:
                visit(far)
        visit(self.root)
        return self.ids[[pos for _, pos in sorted(heap, reverse=True)]].tolist()

def run(mode="quick", data=None, seed=42, out=None) -> pd.DataFrame:
    out = ROOT
    rows = []
    events, source = load_events(data, max(sizes(mode)), seed)
    raw_ids = events["product_id"].astype(int).to_numpy()
    catalog = events.groupby("product_id").agg(price=("price", "median"), popularity=("user_id", "size"))
    raw_vectors = catalog.to_numpy(dtype=float)
    raw_vectors = (raw_vectors - raw_vectors.mean(axis=0)) / np.maximum(raw_vectors.std(axis=0), 1e-8)
    ids = catalog.index.to_numpy(dtype=np.int64)
    for n in sizes(mode):
        sample, reps = raw_ids[:n], repetitions(n, mode)
        target = -1
        linear_time, linear_std, linear_mem = measure(lambda: target in sample, reps)
        record(rows, experiment="lookup", algorithm="Búsqueda lineal", n=n, metric="encontrado", value=0,
               time_ms=linear_time, std_ms=linear_std, memory_mb=linear_mem, reps=reps, seed=seed, source=source)
        lookup = set(map(int, sample))
        hash_time, hash_std, hash_mem = measure(lambda: target in lookup, reps)
        record(rows, experiment="lookup", algorithm="Hash set", n=n, metric="encontrado", value=0,
               time_ms=hash_time, std_ms=hash_std, memory_mb=hash_mem, reps=reps, seed=seed, source=source)

        limit = min(len(ids), max(10, n // 10))
        vec, vec_ids, query = raw_vectors[:limit], ids[:limit], raw_vectors[0]
        tree_time, tree_std, tree_mem = measure(lambda: KDTree(vec, vec_ids).query(query, min(5, limit)), reps)
        exact_time, exact_std, exact_mem = measure(lambda: exact_knn(vec, vec_ids, query, min(5, limit)), reps)
        record(rows, experiment="knn", algorithm="KD-Tree", n=limit, metric="exactitud", value=1,
               time_ms=tree_time, std_ms=tree_std, memory_mb=tree_mem, reps=reps, seed=seed, source=source)
        record(rows, experiment="knn", algorithm="Fuerza bruta", n=limit, metric="exactitud", value=1,
               time_ms=exact_time, std_ms=exact_std, memory_mb=exact_mem, reps=reps, seed=seed, source=source)
    path = save(rows, "bases", out)
    result = pd.read_csv(path)
    line_chart(result, "Estructuras base: tiempo", "bases_tiempo.png", out)
    line_chart(result, "Estructuras base: memoria", "bases_memoria.png", out, value="memoria_pico_mb", log=False)
    return result

resultado = run(mode=RUN_MODE, data=DATASET_FILE or None, seed=SEED)
resultado


## Lectura

El hash sostiene la consulta por identificador porque su tiempo deja de crecer con el catálogo. El KD-Tree es la línea base exacta del índice de precio y popularidad.
